# SolarSDE — Fast Iteration Notebook

Lean loop for tuning the model: **data → closed-form train → all-horizon results + SkyGPT head-to-head.** No rollout (unless `TRAIN_ROLLOUT=True`), no baselines/ablations/CV/economics — those live in `11_final_publication.ipynb`.

**Produces:** all-weather per-horizon CRPS/PICP (STAGE 0) and the SkyGPT exact-cloudy-test per-horizon CRPS/Winkler + h=15 head-to-head vs the published 2.81. ~1.5 h/run on a T4.

Code pulled live from github.com/keshavkrishnan08/SDE.

## 0. Environment + config

In [ ]:
# ==== Setup (fast-iteration: closed-form only, all-horizon + SkyGPT) ====
import os, sys, json, math, time, gc, shutil, subprocess, traceback
from pathlib import Path
import numpy as np, pandas as pd
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
import torch
try:
    import torch._utils, torch._dynamo  # noqa: F401
except Exception as _e:
    print(f"[WARN] dynamo warmup: {_e} — continuing")
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

IN_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kaggle={IN_KAGGLE} Colab={IN_COLAB} device={DEVICE}")
if DEVICE.type != "cuda":
    print("[WARN] No GPU — enable a GPU runtime.")

ROOT = (Path("/kaggle/working") if IN_KAGGLE else Path.cwd()) / "fast_run"
PERSIST_DIR = ROOT / "outputs"; WORK_DIR = ROOT / "work"; DATA_DIR = WORK_DIR / "data"
CHECKPOINT_DIR = PERSIST_DIR / "checkpoints"; RESULTS_DIR = PERSIST_DIR / "results"
LATENT_DIR = PERSIST_DIR / "latents"; SPLITS_DIR = PERSIST_DIR / "splits"
EXTENDED_DIR = PERSIST_DIR / "extended"; FIGURES_DIR = PERSIST_DIR / "figures"
for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR, LATENT_DIR, SPLITS_DIR, EXTENDED_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ===== Config — iterate on these =====
Z_DIM = 64
SKIPPD_VAE_EPOCHS = 12     # CS-VAE (cached after first run if you attach outputs)
CLOSEDFORM_EPOCHS = 60     # the knob you'll mostly tune
TRAIN_ROLLOUT     = False  # set True to also train + ensemble the rollout variant
ARCH = "base"              # which architecture to train + benchmark on SkyGPT.
                           # Options: base | bigmix | wide | deep | gru
                           #   base   K=3 d=128 L=2  (reference closed-form)
                           #   bigmix K=8 d=128 L=3  (heavy-tail mixture for cloudy ramps)
                           #   wide   K=4 d=256 L=2  (wider transformer)
                           #   deep   K=4 d=128 L=4  (deeper transformer)
                           #   gru    GRU encoder    (different temporal bias)
MOTION_GRID = 1            # SPATIAL cloud-motion features (the real beat-attempt):
                           #   1 = global-mean optical flow (4 dims, current)
                           #   3 = 3x3 grid-pooled flow (27 dims) — keeps WHERE clouds
                           #       move toward the sun, the spatial signal SkyGPT uses
                           #   4 = 4x4 grid (48 dims)
                           # Sweep ARCH x MOTION_GRID; if a combo beats 2.81 at h=15,
                           # set the same knobs in notebook 11 for the final paper run.
SEED_ENSEMBLE     = 1      # >1 = deep ensemble: train this many closed-form models
                           # with different seeds and pool their samples. The one
                           # legitimate lever with a real shot at beating SkyGPT
                           # (deep ensembles reliably cut CRPS ~5-12%). Costs
                           # SEED_ENSEMBLE x ~85 min — set 3 when you want the real try.

# Reuse a cached VAE/latents from an attached Kaggle dataset to skip ~25 min.
if IN_KAGGLE and Path("/kaggle/input").exists():
    for ds in Path("/kaggle/input").iterdir():
        if ds.is_dir() and (ds / "checkpoints" / "skippd_vae.pt").exists():
            for sub in ["checkpoints", "latents", "splits", "extended"]:
                s = ds / sub
                if s.exists(): shutil.copytree(s, PERSIST_DIR / sub, dirs_exist_ok=True)
            print(f"  reused cached VAE/latents from {ds.name} — VAE+motion will skip")
print(f"Config: VAE={SKIPPD_VAE_EPOCHS}ep, closed-form={CLOSEDFORM_EPOCHS}ep, rollout={TRAIN_ROLLOUT}")
print(f"PERSIST_DIR={PERSIST_DIR}")


## 1. Pull code from GitHub

In [ ]:
# ==== Pull the SolarSDE codebase from GitHub and import the actual modules ====
# The code that runs below IS the repo code (github.com/keshavkrishnan08/SDE),
# not a copy embedded in this notebook.
REPO_HTTPS = "https://github.com/keshavkrishnan08/SDE.git"
REPO_ZIP   = "https://github.com/keshavkrishnan08/SDE/archive/refs/heads/main.zip"
REPO_DIR = ROOT / "sde_repo"

def _clone_repo():
    if (REPO_DIR / "notebooks" / "_solarsde_v2.py").exists():
        print(f"  repo already present at {REPO_DIR}")
        # refresh to latest main (best effort)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                       capture_output=True, timeout=120)
        return True
    for attempt in range(1, 4):
        try:
            print(f"  git clone (attempt {attempt}) ...")
            r = subprocess.run(["git", "clone", "--depth", "1", REPO_HTTPS, str(REPO_DIR)],
                               capture_output=True, text=True, timeout=300)
            if r.returncode == 0 and (REPO_DIR / "notebooks").exists():
                return True
            print(f"    clone failed: {r.stderr[:200]}")
        except Exception as e:
            print(f"    clone error: {e}")
        time.sleep(5)
    # Fallback: download the repo as a zip archive
    try:
        print("  falling back to zip archive download ...")
        import urllib.request, zipfile, io
        with urllib.request.urlopen(REPO_ZIP, timeout=300) as r:
            zf = zipfile.ZipFile(io.BytesIO(r.read()))
        zf.extractall(ROOT)
        extracted = next(ROOT.glob("SDE-*"))
        if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
        extracted.rename(REPO_DIR)
        return (REPO_DIR / "notebooks").exists()
    except Exception as e:
        print(f"    zip fallback failed: {e}")
        return False

if not _clone_repo():
    raise RuntimeError("Could not obtain the SolarSDE repo from GitHub — check network/repo access.")
MODULE_DIR = REPO_DIR / "notebooks"
sys.path.insert(0, str(MODULE_DIR))
print(f"  modules dir: {MODULE_DIR}")
print(f"  repo modules: {sorted(p.name for p in MODULE_DIR.glob('_*.py'))}")

# ---- Fallback-guarded imports: a missing/broken module never stops the run ----
def _safe_import(module, names):
    out = {}
    try:
        mod = __import__(module, fromlist=names)
        for n in names:
            out[n] = getattr(mod, n)
        print(f"  [OK]   {module}: {len(names)} constants")
    except Exception as e:
        print(f"  [FAIL] {module}: {type(e).__name__}: {str(e)[:120]}")
        for n in names:
            out[n] = f'print("[SKIP] {n} unavailable — module {module} failed to import")'
    return out

globals().update(_safe_import("_master_hardening", ["safe_stage"]))
globals().update(_safe_import("_combined_generator",
    ["SHARED_CODE", "BASELINES_CODE", "STRATIFIED_CODE", "ANALYSIS_CODE"]))
globals().update(_safe_import("_final_generator",
    ["LOAD_DATA_TOLERANT_CODE", "RAMP_AUROC_CODE", "BOOTSTRAP_CIS_CODE",
     "PIT_RELIABILITY_CODE", "ECONOMIC_CAISO_CODE", "LATEX_TABLES_CODE", "ZIP_DOWNLOAD_CODE"]))
globals().update(_safe_import("_colab_master_generator",
    ["CTI_VALIDATION_CODE", "HOLM_BONFERRONI_CODE"]))
globals().update(_safe_import("_skippd_pipeline",
    ["SKIPPD_DOWNLOAD_FULL_CODE", "SKIPPD_PREP_CODE", "SKIPPD_VAE_CODE",
     "SKIPPD_LATENTS_WRITE_CODE", "SKIPPD_HORIZON_OVERRIDE_CODE"]))
globals().update(_safe_import("_solarsde_v2",
    ["MDN_ARCHITECTURE_CODE", "STAGE_0_V2_CODE", "POST_STAGE0_V2_VERIFY_CODE", "ABLATIONS_V2_CODE"]))
globals().update(_safe_import("_solarsde_rollout",
    ["ROLLOUT_ARCH_CODE"]))
globals().update(_safe_import("_skygpt_eval", ["SKYGPT_BENCHMARK_CODE"]))
globals().update(_safe_import("_skygpt_sweep", ["SKYGPT_SWEEP_CODE", "DEEP_ENSEMBLE_CODE"]))
globals().update(_safe_import("_arch_variants", ["ARCH_VARIANTS_CODE"]))
globals().update(_safe_import("_ensemble_eval",
    ["STASH_CLOSEDFORM_CODE", "STASH_ROLLOUT_CODE", "CHAMPION_SELECT_CODE",
     "SKYGPT_TRIPLE_BENCHMARK_CODE"]))
globals().update(_safe_import("_ensemble_eval",
    ["STASH_CLOSEDFORM_CODE", "STASH_ROLLOUT_CODE", "CHAMPION_SELECT_CODE",
     "SKYGPT_TRIPLE_BENCHMARK_CODE"]))
globals().update(_safe_import("_skippd_extras",
    ["IMPLEMENTATION_DETAILS_CODE", "DATA_CARD_CODE", "COMPUTATIONAL_COST_CODE",
     "RELIABILITY_LEVELS_CODE", "SAMPLING_EFFICIENCY_CODE", "ECONOMIC_SENSITIVITY_CODE",
     "CROSS_VALIDATION_V2_CODE"]))

# If safe_stage itself failed to import, provide a minimal local fallback.
if isinstance(globals().get("safe_stage"), str):
    def safe_stage(name, code):
        ind = "\n".join("    " + l if l else "" for l in code.splitlines())
        return (f"try:\n{ind}\nexcept Exception as _e:\n"
                f"    import traceback; traceback.print_exc()\n"
                f"    print('[STAGE FAILED] {name} — continuing.')\n")
    print("  [WARN] using local fallback safe_stage")
print("\nAll modules wired. Code provenance: github.com/keshavkrishnan08/SDE @ main")


## 2. Data: SKIPP'D (~2.3 GB) + SkyGPT test set

In [ ]:
# ==== DOWNLOAD_SKIPPD ====
try:
    exec(safe_stage('DOWNLOAD_SKIPPD', SKIPPD_DOWNLOAD_FULL_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] DOWNLOAD_SKIPPD — continuing to next cell.')


## 3. Preprocess

In [ ]:
# ==== SKIPPD_PREP ====
try:
    exec(safe_stage('SKIPPD_PREP', SKIPPD_PREP_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_PREP — continuing to next cell.')


## 4. CS-VAE + encode + optical-flow motion (skips if cached)

In [ ]:
# ==== SKIPPD_VAE ====
try:
    exec(safe_stage('SKIPPD_VAE', SKIPPD_VAE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_VAE — continuing to next cell.')


## 5. CTI + write contract

In [ ]:
# ==== SKIPPD_WRITE ====
try:
    exec(safe_stage('SKIPPD_WRITE', SKIPPD_LATENTS_WRITE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKIPPD_WRITE — continuing to next cell.')


## 6. Shared + load + 1-min horizon config

In [ ]:
# ==== SHARED ====
try:
    exec(safe_stage('SHARED', SHARED_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SHARED — continuing to next cell.')


In [ ]:
# ==== LOAD_DATA ====
try:
    exec(safe_stage('LOAD_DATA', LOAD_DATA_TOLERANT_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] LOAD_DATA — continuing to next cell.')


In [ ]:
# ==== HORIZON_OVERRIDE ====
try:
    exec(safe_stage('HORIZON_OVERRIDE', SKIPPD_HORIZON_OVERRIDE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] HORIZON_OVERRIDE — continuing to next cell.')


## 7. Architecture (select via ARCH) + train (all-weather per-horizon prints here)

In [ ]:
# ==== CLOSEDFORM_ARCH ====
try:
    exec(safe_stage('CLOSEDFORM_ARCH', MDN_ARCHITECTURE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_ARCH — continuing to next cell.')


In [ ]:
# ==== ARCH_SELECT ====
try:
    exec(safe_stage('ARCH_SELECT', ARCH_VARIANTS_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] ARCH_SELECT — continuing to next cell.')


In [ ]:
# ==== CLOSEDFORM_GLUE ====
try:
    # ClosedFormSDE is set by ARCH_SELECT; fall back to base if that stage was skipped
    if 'ClosedFormSDE' not in globals(): ClosedFormSDE = TemporalLatentSDE
    print('arch ready:', globals().get('ARCH','base'))
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_GLUE — continuing to next cell.')


In [ ]:
# ==== CLOSEDFORM_TRAIN ====
try:
    exec(safe_stage('STAGE0_CLOSEDFORM',
         STAGE_0_V2_CODE.replace('EPOCHS = 60', f'EPOCHS = {CLOSEDFORM_EPOCHS}')), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_TRAIN — continuing to next cell.')


In [ ]:
# ==== CLOSEDFORM_VERIFY ====
try:
    exec(safe_stage('CLOSEDFORM_VERIFY', POST_STAGE0_V2_VERIFY_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] CLOSEDFORM_VERIFY — continuing to next cell.')


## 8. (optional) Rollout variant + ensemble — only if TRAIN_ROLLOUT

In [ ]:
# ==== OPTIONAL_ROLLOUT ====
try:
    if not globals().get('TRAIN_ROLLOUT', False):
        print('[SKIP] TRAIN_ROLLOUT=False — closed-form only (fast path).')
    else:
        exec(safe_stage('STASH_CLOSEDFORM', STASH_CLOSEDFORM_CODE), globals())
        exec(safe_stage('ROLLOUT_ARCH', ROLLOUT_ARCH_CODE), globals())
        exec(safe_stage('STAGE0_ROLLOUT',
             STAGE_0_V2_CODE.replace('EPOCHS = 60', 'EPOCHS = 35')), globals())
        exec(safe_stage('STASH_ROLLOUT', STASH_ROLLOUT_CODE), globals())
        exec(safe_stage('CHAMPION_SELECT', CHAMPION_SELECT_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] OPTIONAL_ROLLOUT — continuing to next cell.')


## 8b. (optional) Deep ensemble — train K seeds (the real lever; SEED_ENSEMBLE>1)

In [ ]:
# ==== DEEP_ENSEMBLE ====
try:
    exec(safe_stage('DEEP_ENSEMBLE', DEEP_ENSEMBLE_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] DEEP_ENSEMBLE — continuing to next cell.')


## 9. SkyGPT exact-benchmark — all horizons + h=15 head-to-head

In [ ]:
# ==== SKYGPT ====
try:
    # triple benchmark if rollout trained, else single-model on the champion
    if globals().get('TRAIN_ROLLOUT', False) and (CHECKPOINT_DIR/'mdn_rollout_best.pt').exists():
        exec(safe_stage('SKYGPT', SKYGPT_TRIPLE_BENCHMARK_CODE), globals())
    else:
        exec(safe_stage('SKYGPT', SKYGPT_BENCHMARK_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKYGPT — continuing to next cell.')


## 9b. Idea SWEEP — post-hoc knobs (val-selected + test-oracle diagnostic)

In [ ]:
# ==== SKYGPT_SWEEP ====
try:
    exec(safe_stage('SKYGPT_SWEEP', SKYGPT_SWEEP_CODE), globals())
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SKYGPT_SWEEP — continuing to next cell.')


## 10. Save results

In [ ]:
# ==== SAVE ====
try:
    import shutil
    out = (Path('/kaggle/working') if IN_KAGGLE else Path.cwd()) / 'fast_results.zip'
    shutil.make_archive(str(out)[:-4], 'zip', RESULTS_DIR)
    print(f'results zipped -> {out}')
    for f in sorted(RESULTS_DIR.glob('*.csv')):
        print(' ', f.name)
except Exception:
    import traceback; traceback.print_exc()
    print('[CELL FAILED] SAVE — continuing to next cell.')
